In [1]:
%matplotlib notebook

In [2]:
from pathlib import Path
from glob import glob
import pandas as pd
import re
import os

In [3]:
import bikipy
from bikipy.preferance.border import ParallelogramBorder

In [4]:
WORKING_DIR = Path("C:/Users/Can/Projects/Neuroscience/bikipy/examples/data/")
DATA_DIR = Path("C:/Users/Can/Projects/Neuroscience/Imen/data/y_maze")
BORDER_IMG_PATH = WORKING_DIR / "images/maze.png"

assert BORDER_IMG_PATH.exists(), f"The image file doesn't exist in {border_img}"
border_img_path_str = str(BORDER_IMG_PATH)

In [5]:
borders = [
        ParallelogramBorder(
            base=[[308.81687801, 193.11825], [288.30224458, 234.14751686]],
            apex=[[174.33205886, 120.17733115], [151.53802172, 158.92719429]],
            guiding_image=border_img_path_str, label="A"
        ),
        ParallelogramBorder(
            base=[[309.95657987, 193.11825], [335.03002072, 234.14751686]],
            apex=[[441.02229344, 116.75822557], [461.53692687, 154.36838686]],
            guiding_image=border_img_path_str, label="B"
        ),
        ParallelogramBorder(
            base=[[290.58164829, 234.14751686], [335.03002072, 234.14751686]],
            apex=[[294.00075387, 394.84547872], [338.44912629, 394.84547872]],
            guiding_image=border_img_path_str, label="C"
        )
    ]

In [6]:
exp_id_finder = re.compile("\d+")

In [7]:
DATA_DIR = Path("C:/Users/Can/Projects/Neuroscience/Imen/data/y_maze")
exp_id_finder = re.compile("\d+")
data_dict = {}
for subdir in os.listdir(str(DATA_DIR)):
    print(subdir)
    for file_path in glob(os.path.join(str(DATA_DIR / subdir), "**.h5")):
        exp_id = exp_id_finder.findall(Path(file_path).stem)[0]
        print(exp_id)
        f = bikipy.DeepLabCutReader.from_hdf(
            file_path, (640, 480), midpoint_groups=(("left_ear", "right_ear"),)
        )
        data_dict[(subdir, exp_id)] = ParallelogramBorder.detect_sequential_border_presence(
            f["mid-left_ear-right_ear"], *borders
        )

after
10
11
12
13
14
15
16
17
18
19
1


c:\users\can\projects\neuroscience\bikipy\bikipy\preferance\border.py:451: UserWarning: Border C has data overlap with other borders
  warn(f"Border {border.label} has data overlap with other borders")


20
21
22
23
24
25
26
27
28
29
2
30
31
32
33
34
35
36
37
38
39
3
40
41
42
43
4
5
6
7
8
9
before
10
11
12
13
14
15
16
17
18
19
1
20
21
22
23
24
25
26
27
28
29
2
30
31
32
33
34
35
36
37
38
39
3
40
41
42
43
44
45
46
4
5
6
7
8
9


In [8]:
reduced = {}
for info, data in data_dict.items():
    print(info)
    length = len(data)
    i = 1
    current_char = data[0]
    reduced_data = [current_char]
    while i < length:
        if data[i] and current_char != data[i]:
            current_char = data[i]
            reduced_data.append(current_char)
        i += 1
    reduced[info] = reduced_data

('after', '10')
('after', '11')
('after', '12')
('after', '13')
('after', '14')
('after', '15')
('after', '16')
('after', '17')
('after', '18')
('after', '19')
('after', '1')
('after', '20')
('after', '21')
('after', '22')
('after', '23')
('after', '24')
('after', '25')
('after', '26')
('after', '27')
('after', '28')
('after', '29')
('after', '2')
('after', '30')
('after', '31')
('after', '32')
('after', '33')
('after', '34')
('after', '35')
('after', '36')
('after', '37')
('after', '38')
('after', '39')
('after', '3')
('after', '40')
('after', '41')
('after', '42')
('after', '43')
('after', '4')
('after', '5')
('after', '6')
('after', '7')
('after', '8')
('after', '9')
('before', '10')
('before', '11')
('before', '12')
('before', '13')
('before', '14')
('before', '15')
('before', '16')
('before', '17')
('before', '18')
('before', '19')
('before', '1')
('before', '20')
('before', '21')
('before', '22')
('before', '23')
('before', '24')
('before', '25')
('before', '26')
('before', '27')

In [9]:
spontaneous_alterntations = {}
for info, data in reduced.items():
    max_alternations = len(data) - 2
    alternations = 0
    for i in range(max_alternations):
        current_string = f"{data[i]}{data[i+1]}{data[i+2]}"
        if "A" in current_string and "B" in current_string and "C" in current_string:
            alternations += 1
    spontaneous_alterntations[info] = 100 * alternations / max_alternations

In [12]:
pd.DataFrame.from_dict(spontaneous_alterntations, "index", columns=["Spontaneous Alterntations"]).to_excel("Spontaneous Alterntations.xlsx")